In [1]:
## General imports
import numpy as np
## Local imports
import mapLearningUtils as mlu
import get_data_matrix as getMat
import train_MappingJM as trMap

In [ ]:
## Set configuration
# Data handling
src_data       = ['CALIB', 'ASSIST']
data_type_list = ['exoPositions', 'exoVelocities', 'humanPositions', 'humanVelocities', 'wristSensor']
list_vars      = ['q3', 'q4', 'qd3', 'qd4', 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz', 'l_a', 'l_fa',
                  'shoulder_elv', 'elbow_flexion', 'dshoulder_elv', 'delbow_flexion', 'trial', 'subj']
# CALIB params
path_calib         = './data_CALIB'
name_expe          = 'CALIB'
save_path_calib    = './data_CALIB/dataMatCalib.pkl'
nb_subj            = 17
cond               = 'MJ'
nb_trials_calib    = 10
duration_idx_calib = 2900
params_calib       = {'path': path_calib, 'expeName': name_expe, 'save_path': save_path_calib, 'nb_subj': nb_subj, 'condition': cond,
                      'nb_trials': nb_trials_calib, 'dataTypes': data_type_list, 'listOfVariables': list_vars, 'durationIndex': duration_idx_calib}
# ASSIST params
path_assist         = './data_ASSIST'
name_expe           = 'ASSIST'
save_path_assist    = './data_ASSIST/dataMatAssist.pkl'
subjs_list          = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S09', 'S10', 'S12', 'S14']
conds_list          = ['T', 'ES', 'EG']
phases_list         = ['PE', 'PS', 'RE', 'RS']
nb_trials_calib     = 26
duration_idx_assist = 500
params_assist       = {'path': path_assist, 'expeName': name_expe, 'save_path': save_path_assist, 'subjList': subjs_list, 'conditions': conds_list,
                      'nb_trials': nb_trials_calib, 'phases': phases_list, 'dataTypes': data_type_list, 'listOfVariables': list_vars,
                      'durationIndex': duration_idx_assist}

In [8]:
## Get data matrices
# Import CALIB data
data_calib = getMat.get_data(src_data[0], params_calib)
print('CALIB data shape: ' + str(data_calib.shape))
# Import ASSIST data
data_assist = getMat.get_data(src_data[1], params_assist)
print('ASSIST data shape: ' + str(data_assist.shape))

Loaded calibration data from pickle.
CALIB data shape: (493000, 18)
Loaded assistance data from pickle.
ASSIST data shape: (227322, 18)


In [9]:
## Pre-processing parameters
ablationsList = ['velocity', 'anthropo', 'forcestorques', 'forces', 'torques', 'Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
foldsList     = ['2', '5', 'LOO']
forceLoc = 'wrist'

params_prepro = {'ablations': ablationsList, 'folds': foldsList, 'forceLoc': forceLoc}

In [ ]:
## Prepare dictionnary with all ablations and foldings
fold_dict_all = mlu.get_all_folded_ablated_data(data_calib, data_assist, params_prepro)

In [ ]:
## Mapping parameters
degree     = 3 # Degree of multivariate polynomial regression
nb_modes   = 8
nb_hChan   = 64
nb_maxIter = 500
limLoss    = 1e-6

params_MVLR = {"nb_folds": nb_folds}
params_MVPR = {"nb_folds": nb_folds, "degree": degree}
params_FNO = {"nb_folds": nb_folds, "nb_modes": nb_modes, "nb_hiddenChannels": nb_hChan,
              "nb_maxTrainingIterations": nb_maxIter, "limitConvergenceLoss": limLoss}

In [ ]:
## Train models
models_MVLR = trMap.train_MappingJM(fold_dict, "MVLR", params_MVLR)
models_MVPR = trMap.train_MappingJM(fold_dict, "MVPR", params_MVPR)
models_FNO = trMap.train_MappingJM(fold_dict, "MVPR", params_FNO)

{'fold1': LinearRegression(), 'fold2': LinearRegression()}
